# GTCN-G style model on the paper's data subset, with ablations

The paper's own model, GTCN-G, has no code, so this is our implementation of the parts the paper describes, on the same data subset and 5:2:3 split as the subset replication notebook (flows of UNSW-NB15_1.csv, 637,891 flows after our duplicate removal).

What follows the paper: flows are the nodes of the line graph (two flows are neighbours when they share an endpoint, an (IP, port) pair), the neighbours of a flow are sampled per mini-batch of 500, 6-head attention with dropout 0.5 on the attention coefficients, a residual branch with a linear transform of the flow's own features, a gated temporal convolution over the neighbours and a graph convolution branch.

What we had to choose or simplify, because the paper does not say or the original does not scale:
- One hop of neighbours (8 sampled per endpoint, 16 in total); the number of layers is not given in the paper.
- The temporal sequence is the sampled neighbours ordered by their position in the raw file (flows are stored in time order); the paper does not describe how its sequences are built.
- The graph convolution branch is a mean over the neighbours. The paper's adaptive adjacency matrix learns an N x N embedding product, which is not feasible for hundreds of thousands of flows.
- Plain cross-entropy, Adam, learning rate 0.001 with halving on a plateau (the paper's loss and optimizer are not stated; 0.007 collapsed our baseline).

Ablations switch one part off at a time: residual branch, attention (replaced by a mean), temporal branch, graph convolution branch, all neighbours (isolated flows) and real neighbours replaced by random flows.

In [ ]:
import copy

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

DEBUG = False
SEEDS = [42] if DEBUG else [42, 43, 44]
RUN = ["full", "no residual", "mean instead of attention", "no temporal branch", "no graph convolution branch", "isolated flows", "shuffled neighbours"]
PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONDITIONS = {
    "full": {},
    "no residual": {"residual": False},
    "mean instead of attention": {"attn": False},
    "no temporal branch": {"temporal": False},
    "no graph convolution branch": {"spatial": False},
    "isolated flows": {"graph": "none"},
    "shuffled neighbours": {"graph": "shuffled"},
}
HEADS, HEAD_DIM = 6, 24
FANOUT = 8
K = 2 * FANOUT
BRANCH = 64
DROPOUT = 0.2
ATTN_DROPOUT = 0.5
LR = 0.001
BATCH = 500
PAPER_EPOCHS = 10
MAX_EPOCHS = 3 if DEBUG else 60
PATIENCE = 3 if DEBUG else 10

## 1. Data and 5:2:3 split

In [ ]:
df = pd.concat([pd.read_parquet(f"{PATH}/train.parquet"), pd.read_parquet(f"{PATH}/test.parquet")])
df = df[df.index < 700_001]
graph = pd.concat([pd.read_csv(f"{PATH}/train_graph.csv", index_col="row_id"),
                   pd.read_csv(f"{PATH}/test_graph.csv", index_col="row_id")]).loc[df.index]
if DEBUG:
    df = df.sample(200_000, random_state=0)
    graph = graph.loc[df.index]

class_names = list(joblib.load(f"{PATH}/preprocessing.pkl")["label_encoder"].classes_)
n_classes = len(class_names)
feature_cols = [c for c in df.columns if c != "label"]

train_ids, rest_ids = train_test_split(df.index, train_size=0.5, stratify=df["label"], random_state=0)
val_ids, test_ids = train_test_split(rest_ids, train_size=0.4, stratify=df.loc[rest_ids, "label"], random_state=0)
print({k: len(v) for k, v in [("train", train_ids), ("val", val_ids), ("test", test_ids)]}, "| features", len(feature_cols))

## 2. Graphs

Nodes are (IP, port) endpoints and flows are edges, so the line graph's nodes are flows. For every endpoint the flows that touch it are stored as a neighbour list (CSR); the neighbours of a flow are the flows at its two endpoints. Each split gets its own graph.

In [ ]:
def build_graph(ids):
    feat_df, graph_df = df.loc[ids], graph.loc[ids]
    n = len(feat_df)
    ip_code, _ = pd.factorize(pd.concat([graph_df["srcip"], graph_df["dstip"]]))
    ports = np.concatenate([graph_df["sport"].fillna(-1), graph_df["dsport"].fillna(-1)]).astype("int64") + 1
    node_id, uniq = pd.factorize(ip_code.astype("int64") * 70_000 + ports)
    node_id = torch.from_numpy(node_id).to(DEVICE)
    src, dst = node_id[:n], node_id[n:]

    nodes = torch.cat([src, dst])
    flow = torch.cat([torch.arange(n, device=DEVICE)] * 2)
    order = torch.argsort(nodes)
    ptr = torch.zeros(len(uniq) + 1, dtype=torch.long, device=DEVICE)
    ptr[1:] = torch.cumsum(torch.bincount(nodes, minlength=len(uniq)), 0)
    return {"src": src, "dst": dst, "eflow": flow[order], "ptr": ptr,
            "rid": torch.from_numpy(feat_df.index.to_numpy("int64")).to(DEVICE),
            "x": torch.from_numpy(feat_df[feature_cols].to_numpy("float32")).to(DEVICE),
            "y": torch.from_numpy(feat_df["label"].to_numpy("int64")).to(DEVICE)}


G = {"train": build_graph(train_ids), "val": build_graph(val_ids), "test": build_graph(test_ids)}
for name, g in G.items():
    deg = torch.diff(g["ptr"]).float()
    print(f"{name:<6} flows {len(g['src']):>9,} | endpoints {len(deg):>9,} | mean flows per endpoint {deg.mean():.2f}")

## 3. Model

For a batch of flows the model draws 8 neighbouring flows from each endpoint (16 in total, the flow itself excluded), orders them by time and feeds four branches: attention over the neighbours, a residual branch with the flow's own features, a gated temporal convolution over the ordered neighbours and a mean graph convolution. The branch outputs are concatenated and classified. A flow whose endpoints carry no other flow has no neighbours; its neighbour branches are zero and only the residual branch speaks.

In [ ]:
def neighbours(g, idx, mode):
    b = len(idx)
    if mode == "none":
        return torch.zeros(b, K, dtype=torch.long, device=DEVICE), torch.zeros(b, K, dtype=torch.bool, device=DEVICE)
    if mode == "shuffled":
        return torch.randint(0, len(g["src"]), (b, K), device=DEVICE), torch.ones(b, K, dtype=torch.bool, device=DEVICE)
    ends = torch.cat([g["src"][idx], g["dst"][idx]])
    start = g["ptr"][ends]
    deg = g["ptr"][ends + 1] - start
    pos = start[:, None] + (torch.rand(2 * b, FANOUT, device=DEVICE) * deg[:, None]).long()
    flows = g["eflow"][pos]
    flows = torch.cat([flows[:b], flows[b:]], 1)
    return flows, flows != idx[:, None]


class GTCNG(nn.Module):
    def __init__(self, n_feat, n_classes, residual=True, attn=True, temporal=True, spatial=True, graph="real"):
        super().__init__()
        self.residual, self.attn, self.temporal, self.spatial, self.graph = residual, attn, temporal, spatial, graph
        width = HEADS * HEAD_DIM
        self.proj = nn.Linear(n_feat, width, bias=False)
        self.a_nbr = nn.Parameter(torch.randn(HEADS, HEAD_DIM) * 0.1)
        self.a_self = nn.Parameter(torch.randn(HEADS, HEAD_DIM) * 0.1)
        self.attn_drop = nn.Dropout(ATTN_DROPOUT)
        if residual:
            self.res = nn.Linear(n_feat, BRANCH)
            width += BRANCH
        if temporal:
            self.t_in = nn.Linear(n_feat, BRANCH)
            self.t_conv = nn.ModuleList([nn.Conv1d(BRANCH, 2 * BRANCH, 3, dilation=d) for d in (1, 2)])
            width += BRANCH
        if spatial:
            self.sp = nn.Linear(n_feat, BRANCH)
            width += BRANCH
        self.head = nn.Sequential(nn.Linear(width, 128), nn.ReLU(), nn.Dropout(DROPOUT), nn.Linear(128, n_classes))

    def forward(self, g, idx):
        b = len(idx)
        x_self = g["x"][idx]
        flows, mask = neighbours(g, idx, self.graph)
        order = torch.where(mask, g["rid"][flows], torch.full_like(flows, -1)).argsort(1)
        flows, mask = flows.gather(1, order), mask.gather(1, order)
        m = mask.float()
        x_nbr = g["x"][flows] * m[..., None]
        count = m.sum(1).clamp(min=1)

        w_nbr = self.proj(x_nbr).view(b, K, HEADS, HEAD_DIM)
        if self.attn:
            w_self = self.proj(x_self).view(b, HEADS, HEAD_DIM)
            score = F.leaky_relu((w_nbr * self.a_nbr).sum(-1) + (w_self * self.a_self).sum(-1)[:, None])
            alpha = torch.softmax(score.masked_fill(~mask[..., None], -1e9), dim=1) * m[..., None]
            agg = (self.attn_drop(alpha)[..., None] * w_nbr).sum(1)
        else:
            agg = (w_nbr * m[..., None, None]).sum(1) / count[:, None, None]
        parts = [F.elu(agg).flatten(1)]

        if self.residual:
            parts.append(F.relu(self.res(x_self)))
        if self.temporal:
            h = (self.t_in(x_nbr) * m[..., None]).transpose(1, 2)
            for conv, dilation in zip(self.t_conv, (1, 2)):
                gate, value = conv(F.pad(h, (2 * dilation, 0))).chunk(2, 1)
                h = torch.tanh(value) * torch.sigmoid(gate) + h
            parts.append(h[:, :, -1])
        if self.spatial:
            parts.append(F.relu(self.sp(x_nbr.sum(1) / count[:, None])))
        return self.head(torch.cat(parts, 1))


loss_fn = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model, g, batch=5000):
    model.eval()
    logits = torch.cat([model(g, torch.arange(i, min(i + batch, len(g["src"])), device=DEVICE))
                        for i in range(0, len(g["src"]), batch)])
    return loss_fn(logits, g["y"]).item(), logits.argmax(1).cpu().numpy()


def scores(y_true, pred):
    per_class = f1_score(y_true, pred, labels=np.arange(n_classes), average=None, zero_division=0)
    return {"weighted F1": f1_score(y_true, pred, average="weighted", zero_division=0),
            "macro F1": per_class.mean(), "accuracy": accuracy_score(y_true, pred)}, per_class

## 4. Training

Same protocol as the baseline notebook: batches of 500, early stopping on validation weighted F1 (patience 10), learning rate halved after 5 epochs without improvement, at most 60 epochs. Test scores are reported after epoch 10 (where the paper stops) and at the best validation epoch.

In [ ]:
def run(condition, seed):
    torch.manual_seed(seed)
    model = GTCNG(len(feature_cols), n_classes, **CONDITIONS[condition]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4)
    g_train, y_val, y_test = G["train"], G["val"]["y"].cpu().numpy(), G["test"]["y"].cpu().numpy()
    n_train = len(g_train["src"])

    history, best, bad, at_paper = [], {"f1": -1.0}, 0, None
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=DEVICE)
        total = torch.zeros((), device=DEVICE)
        for i in range(0, n_train, BATCH):
            idx = perm[i:i + BATCH]
            loss = loss_fn(model(g_train, idx), g_train["y"][idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.detach() * len(idx)

        val_loss, val_pred = evaluate(model, G["val"])
        val, _ = scores(y_val, val_pred)
        history.append({"condition": condition, "seed": seed, "epoch": epoch, "train loss": total.item() / n_train,
                        "val loss": val_loss, "val weighted F1": val["weighted F1"], "val macro F1": val["macro F1"]})
        print(f"{condition} | seed {seed} epoch {epoch:>3} | train loss {history[-1]['train loss']:.4f} | val loss {val_loss:.4f} "
              f"| val weighted F1 {val['weighted F1']:.4f} | val macro F1 {val['macro F1']:.4f}", flush=True)

        sched.step(val["weighted F1"])
        if epoch == PAPER_EPOCHS:
            at_paper, _ = scores(y_test, evaluate(model, G["test"])[1])
        if val["weighted F1"] > best["f1"]:
            best, bad = {"f1": val["weighted F1"], "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best["state"])
    test_pred = evaluate(model, G["test"])[1]
    at_best, per_class = scores(y_test, test_pred)
    if at_paper is None:
        at_paper = at_best
    row = {"condition": condition, "seed": seed, "best epoch": best["epoch"], "last epoch": epoch,
           **{f"{k} (epoch {PAPER_EPOCHS})": v for k, v in at_paper.items()},
           **{f"{k} (best epoch)": v for k, v in at_best.items()}}
    return pd.DataFrame(history), row, {"condition": condition, "seed": seed, **dict(zip(class_names, per_class))}, test_pred.astype("int8")


histories, rows, class_rows, preds = [], [], [], {}
for condition in RUN:
    for seed in SEEDS:
        history, row, class_row, pred = run(condition, seed)
        histories.append(history)
        rows.append(row)
        class_rows.append(class_row)
        preds[f"{condition}|{seed}"] = pred
        pd.concat(histories).to_csv("/kaggle/working/gtcng_history.csv", index=False)
        pd.DataFrame(rows).to_csv("/kaggle/working/gtcng_runs.csv", index=False)
        pd.DataFrame(class_rows).to_csv("/kaggle/working/gtcng_per_class.csv", index=False)
history = pd.concat(histories)
results = pd.DataFrame(rows)
class_results = pd.DataFrame(class_rows)
pd.DataFrame(preds, index=test_ids).to_parquet("/kaggle/working/gtcng_test_pred.parquet")

## 5. Training curves

In [ ]:
if "full" in RUN:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for ax, col, title in zip(axes, ["train loss", "val loss"], ["Training loss", "Validation loss"]):
        sns.lineplot(data=history[history["condition"] == "full"], x="epoch", y=col, hue="seed", palette="deep", ax=ax)
        ax.axvline(PAPER_EPOCHS, color="gray", linestyle="--")
        ax.set_title(f"{title} (full model)")
    plt.tight_layout()
    plt.savefig("/kaggle/working/fig_01.png", dpi=130, bbox_inches="tight")
    plt.show()

## 6. Test results

In [ ]:
metrics = ["weighted F1 (epoch 10)", "macro F1 (epoch 10)", "weighted F1 (best epoch)", "macro F1 (best epoch)"]
summary = results.groupby("condition", sort=False)[metrics].agg(["mean", "std"])
print(summary.round(4).to_string())
summary.to_csv("/kaggle/working/gtcng_summary.csv")
print(class_results.drop(columns="seed").groupby("condition", sort=False).mean().round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col in zip(axes, ["weighted F1 (best epoch)", "macro F1 (best epoch)"]):
    sns.barplot(data=results, y="condition", x=col, errorbar="sd", color="steelblue", ax=ax)
    ax.set_title(f"Test {col}")
axes[0].set_xlim(0.95, 1.0)
plt.tight_layout()
plt.savefig("/kaggle/working/fig_02.png", dpi=130, bbox_inches="tight")
plt.show()